# 096 — Generación 3D y mundos sintéticos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**NeRF**: la escena es una función aprendida `F(x, d) → (c, σ)` — posición x y
dirección de vista d producen color c y densidad σ (la densidad solo depende de x: la
geometría no cambia con el observador). Render volumétrico por rayo con N muestras:

```text
αᵢ = 1 − exp(−σᵢ·δᵢ)          opacidad del segmento
Tᵢ = exp(−Σ_{j<i} σⱼ·δⱼ)      transmitancia (luz que llega hasta i)
C  = Σᵢ Tᵢ·αᵢ·cᵢ               color del píxel
```

Todo es diferenciable: se entrena minimizando el error fotométrico contra fotos
calibradas. Costo: cientos de evaluaciones del MLP por píxel.

**3D Gaussian Splatting**: sustituye el campo implícito por millones de gaussianas 3D
explícitas {μ, Σ, α, color}; el render proyecta cada gaussiana a la imagen, ordena por
profundidad y compone con alpha-blending rasterizado → entrenamiento en minutos y
render >100 fps, a cambio de mucha más memoria y heurísticas de densificación.

**DreamFusion / SDS**: genera 3D desde texto sin datos 3D — renderiza el NeRF desde
cámaras aleatorias y usa un modelo de difusión 2D congelado como crítico (Score
Distillation Sampling). Limitaciones: sobre-saturación y el problema *Janus* (caras
repetidas), porque el crítico 2D no sabe desde dónde mira.


## 🧮 Ejemplo de referencia

3 muestras con δ = 1 y σ = (0.5, 1.0, 2.0): α = (0.3935, 0.6321, 0.8647);
T = (1.0, e⁻⁰·⁵ = 0.6065, e⁻¹·⁵ = 0.2231); pesos w = (0.3935, 0.3834, 0.1929).
Con c₁=(1,0,0), c₂=(0,1,0), c₃=(0,0,1): **C = (0.394, 0.383, 0.193)**. La muestra más
densa (σ=2) pesa MENOS que las anteriores: la materia de delante ya absorbió el 78 %
del rayo. Reprodúcelo a mano antes de ejecutar el laboratorio.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=96)
show(result)


## Reflexión

1. ¿Por qué NeRF hace que σ dependa solo de la posición x pero el color c dependa
   también de la dirección d, y qué efecto visual sería imposible si c ignorara d?
2. El problema Janus de DreamFusion (caras repetidas alrededor del objeto): ¿por qué
   es una consecuencia directa de usar un crítico 2D por vista, y qué información
   adicional lo mitigaría?
3. 3DGS entrena en minutos y renderiza a >100 fps donde NeRF tarda horas y segundos
   por frame: ¿qué se paga a cambio (memoria, edición, heurísticas) y en qué
   aplicación seguiría prefiriendo la representación implícita?
